# Pretrained DDPM as a K-sweep baseline (cross-family universality test)

**Question:** does a *real pretrained diffusion model* (`google/ddpm-cifar10-32`, ~35.7M params)
obey the same inference-depth law $K^*(\sigma) \approx C\sigma^\alpha$ as our micro EBMs
(\u03b1 = 1.376 \u00b1 0.003) and the 77K direct-score net (\u03b1 = 1.376 \u00b1 0.002)?
If yes, the law spans 5.8K\u219235M params and amateur\u2192production training \u2014 the
cross-family claim becomes unimpeachable (reviewer M1 fully retired).

**Protocol provenance:** this notebook is a *verbatim mirror* of the harness in
`theory/tier0_seeds.py` / `theory/phase_b_diffusion.py` (per-image continuous $K^*$,
`SIGMAS`, `K_MAX=30`, `dt=0.05`, `decay=0.97`, clamped score, eval on the first 64
CIFAR-10 test images, 2 eval seeds). **Do not edit the PROTOCOL cell** \u2014 comparability
with the existing \u03b1 numbers depends on it.

**Two conditioning variants:**
- **V1 (primary, apples-to-apples):** condition the UNet at fixed $t^*$ matching
  $\sigma=0.15$ for every step \u2014 the exact analog of our single-$\sigma_{train}$ EBM recipe.
- **V2 (secondary):** condition at the $t$ matching each eval noise level $\sigma_0$ \u2014
  tests whether the law survives noise-matched conditioning.

**Safety:** Drive is mounted first; results JSON is written to Drive **after every \u03c3**
and the run **auto-resumes** from it \u2014 you can close the laptop, lose the runtime,
and just hit *Run all* again.

Runtime: GPU (any \u2014 T4 is enough; ~10\u201320 min total). Runtime \u2192 Change runtime type \u2192 GPU.

In [ ]:
# ============ 1. Mount Google Drive FIRST (all results stream here) ============
from google.colab import drive
drive.mount('/content/drive')

import json, time
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/AAAI_kstar/ddpm_baseline')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def save_json(obj, name):
    """Atomic-ish write to Drive so progress survives disconnects mid-write."""
    p = DRIVE_DIR / name
    tmp = p.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2))
    tmp.replace(p)
    print(f'[saved -> Drive] {p.name}  ({time.strftime("%H:%M:%S")})')

def load_json(name, default):
    p = DRIVE_DIR / name
    return json.loads(p.read_text()) if p.exists() else default

print('Drive ready:', DRIVE_DIR)

In [ ]:
# ============ 2. Installs + imports ============
%pip install -q diffusers accelerate

import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device,
      f'[{torch.cuda.get_device_name(0)}]' if device.type == 'cuda' else '(CPU \u2014 switch runtime to GPU!)')

In [ ]:
# ============ 3. PROTOCOL (verbatim mirror of theory/tier0_seeds.py \u2014 DO NOT EDIT) ============
SIGMAS = [0.05, 0.08, 0.12, 0.16, 0.20, 0.25, 0.30]
K_MAX  = 30
DT     = 0.05
DECAY  = 0.97
N_EVAL = 64
EVAL_SEEDS = 2
SIGMA_TRAIN_REF = 0.15   # the EBM/score recipe's single training sigma

def psnr_per_image(u, clean):
    mse = F.mse_loss(u.clamp(-1, 1), clean, reduction='none').mean(dim=[1, 2, 3])
    return (10 * torch.log10(4.0 / mse)).cpu().numpy()   # (N,)  peak=4.0 for [-1,1] data

def fit_alpha(sigmas, kstars):
    sigmas, kstars = np.array(sigmas, float), np.array(kstars, float)
    ok = kstars > 0
    if ok.sum() < 4 or np.std(kstars[ok]) < 1e-6:
        return float('nan'), float('nan')
    x, y = np.log(sigmas[ok]), np.log(kstars[ok])
    slope, intercept = np.polyfit(x, y, 1)
    yh = intercept + slope * x
    r2 = 1 - np.sum((y - yh) ** 2) / np.sum((y - y.mean()) ** 2)
    return float(slope), float(r2)

print('protocol locked:', dict(SIGMAS=SIGMAS, K_MAX=K_MAX, DT=DT, DECAY=DECAY, N_EVAL=N_EVAL))

In [ ]:
# ============ 4. Eval data: first 64 CIFAR-10 TEST images in [-1,1] (mirrors tier0) ============
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
test_ds = datasets.CIFAR10('/content/data', train=False, download=True, transform=tf)
clean = torch.stack([test_ds[i][0] for i in range(N_EVAL)]).to(device)
print('eval set:', tuple(clean.shape), 'range', float(clean.min()), float(clean.max()))

In [ ]:
# ============ 5. Pretrained DDPM + VP\u2192VE score conversion ============
# The UNet predicts eps(x_t, t) under the VP forward process x_t = sqrt(abar)x0 + sqrt(1-abar)eps.
# Our harness corrupts additively (VE view): u = x0 + sigma*eps. SNR-matching the two gives
#   sigma_t = sqrt(1-abar_t)/sqrt(abar_t),   x_t = sqrt(abar_t) * u,
# and the score of u is   s(u) = grad_u log p(u) = -eps_hat(sqrt(abar_t)*u, t) / sigma_t.
from diffusers import UNet2DModel, DDPMScheduler

MODEL_ID = 'google/ddpm-cifar10-32'
unet = UNet2DModel.from_pretrained(MODEL_ID).to(device).eval()
sched = DDPMScheduler.from_pretrained(MODEL_ID)
abar = sched.alphas_cumprod.to(device)                       # (T,)
sigma_vp = ((1 - abar) / abar).sqrt()                        # VE-equivalent sigma per t

def t_for_sigma(sig):
    return int(torch.argmin((sigma_vp - sig).abs()).item())

@torch.no_grad()
def score_at(u, t):
    a = abar[t].sqrt()
    tt = torch.full((u.shape[0],), t, device=u.device, dtype=torch.long)
    eps = unet(a * u, tt).sample
    return -eps / sigma_vp[t]

n_params = sum(p.numel() for p in unet.parameters())
print(f'{MODEL_ID}: {n_params:,} params')
print(f'sigma_vp range: [{float(sigma_vp.min()):.4f}, {float(sigma_vp.max()):.1f}]')
for s in [0.05, 0.15, 0.30]:
    t = t_for_sigma(s)
    print(f'  sigma={s:.2f} -> t={t}  (sigma_t={float(sigma_vp[t]):.4f})')

In [ ]:
# ============ 6. Sanity check: a few protocol steps must RAISE PSNR at sigma=0.15 ============
torch.manual_seed(0)
u = clean[:8] + 0.15 * torch.randn_like(clean[:8])
t015 = t_for_sigma(SIGMA_TRAIN_REF)
traj, step = [psnr_per_image(u, clean[:8]).mean()], DT
for k in range(10):
    u = u + step * score_at(u, t015).clamp(-1, 1)   # clamped score: mirrors EBM's clamped gradient
    step *= DECAY
    traj.append(psnr_per_image(u, clean[:8]).mean())
print('PSNR trajectory (sigma=0.15):', [round(float(x), 2) for x in traj])
assert max(traj[1:]) > traj[0] + 1.0, 'PSNR did not improve \u2014 check the score conversion!'
print('OK \u2014 pretrained DDPM denoises under the protocol step rule.')

In [ ]:
# ============ 7. K-sweep core (mirrors phase_b_diffusion.kstar_score) ============
def kstar_ddpm(clean, sigma, t_cond):
    """Per-image continuous K* under clamped score-ascent, conditioning fixed at t_cond."""
    per_image, mean_curves = [], []
    for es in range(EVAL_SEEDS):
        torch.manual_seed(10_000 + es)            # same eval-noise seeds as tier0
        u = clean + sigma * torch.randn_like(clean)
        series = [psnr_per_image(u, clean)]
        step = DT
        for _ in range(K_MAX):
            u = u + step * score_at(u, t_cond).clamp(-1, 1)
            step *= DECAY
            series.append(psnr_per_image(u, clean))
        arr = np.stack(series, axis=0)            # (K+1, N)
        per_image.append(arr.argmax(axis=0))      # per-image integer K*
        mean_curves.append(arr.mean(axis=1).tolist())
    return float(np.mean(per_image)), mean_curves

In [ ]:
# ============ 8. V1 (PRIMARY): fixed conditioning at sigma=0.15 \u2014 saves to Drive after each sigma ============
RESULTS_NAME = 'ddpm_results.json'
res = load_json(RESULTS_NAME, {'model': MODEL_ID, 'n_params': n_params,
                               'protocol': dict(SIGMAS=SIGMAS, K_MAX=K_MAX, DT=DT,
                                                DECAY=DECAY, N_EVAL=N_EVAL,
                                                EVAL_SEEDS=EVAL_SEEDS),
                               'V1_fixed_cond_0.15': {}, 'V2_noise_matched': {}})

t015 = t_for_sigma(SIGMA_TRAIN_REF)
for sig in SIGMAS:
    key = f'{sig:.2f}'
    if key in res['V1_fixed_cond_0.15']:
        print(f'[skip] V1 sigma={key} (K*={res["V1_fixed_cond_0.15"][key]["kstar"]:.2f})')
        continue
    t0 = time.time()
    k, curves = kstar_ddpm(clean, sig, t015)
    res['V1_fixed_cond_0.15'][key] = {'kstar': k, 't_cond': t015, 'mean_psnr_curves': curves}
    print(f'V1 sigma={key}:  K*={k:.2f}   [{time.time()-t0:.0f}s]')
    save_json(res, RESULTS_NAME)   # <- progress persists even if the runtime dies here
print('V1 complete.')

In [ ]:
# ============ 9. V2 (SECONDARY): noise-matched conditioning t = t(sigma_0) ============
for sig in SIGMAS:
    key = f'{sig:.2f}'
    if key in res['V2_noise_matched']:
        print(f'[skip] V2 sigma={key} (K*={res["V2_noise_matched"][key]["kstar"]:.2f})')
        continue
    t_cond = t_for_sigma(sig)
    t0 = time.time()
    k, curves = kstar_ddpm(clean, sig, t_cond)
    res['V2_noise_matched'][key] = {'kstar': k, 't_cond': t_cond, 'mean_psnr_curves': curves}
    print(f'V2 sigma={key} (t={t_cond}):  K*={k:.2f}   [{time.time()-t0:.0f}s]')
    save_json(res, RESULTS_NAME)
print('V2 complete.')

In [ ]:
# ============ 10. Fit alpha, compare to the universality references, save summary ============
REFERENCES = {  # from tier0_seeds.py / phase_b_diffusion.py (matched recipe, CIFAR-10)
    'EBM heads (KAN/GELU/SiLU/Tanh, 32K, trained)': '1.376 +/- 0.003',
    'Direct score net (77K, trained)':              '1.376 +/- 0.002',
}
summary = {}
for variant in ('V1_fixed_cond_0.15', 'V2_noise_matched'):
    d = res[variant]
    if len(d) < 4:
        print(f'{variant}: only {len(d)} sigmas done \u2014 run the cells above.')
        continue
    sigs = sorted(float(s) for s in d)
    ks = [d[f'{s:.2f}']['kstar'] for s in sigs]
    alpha, r2 = fit_alpha(sigs, ks)
    summary[variant] = {'alpha': alpha, 'r2': r2, 'kstars': dict(zip(map(str, sigs), ks))}
    print(f'{variant}:  K*={[round(k, 2) for k in ks]}')
    print(f'  -> alpha = {alpha:.3f}   R^2 = {r2:.3f}')
print('\nreference exponents (same protocol):')
for k, v in REFERENCES.items():
    print(f'  {k}: alpha = {v}')
res['summary'] = summary
save_json(res, RESULTS_NAME)

In [ ]:
# ============ 11. Figure: K*(sigma) log-log with power-law fits \u2014 saved to Drive ============
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.0, 3.8))
styles = {'V1_fixed_cond_0.15': ('o-', 'V1: fixed cond. $\\sigma{=}0.15$'),
          'V2_noise_matched':   ('s--', 'V2: noise-matched cond.')}
for variant, (st, lbl) in styles.items():
    if variant not in summary:
        continue
    s = summary[variant]
    sigs = np.array([float(x) for x in s['kstars']])
    ks = np.array(list(s['kstars'].values()))
    ax.loglog(sigs, ks, st, ms=4.5,
              label=f"{lbl}: $\\alpha$={s['alpha']:.2f} (R\u00b2={s['r2']:.3f})")
ax.set_xlabel('noise level $\\sigma$')
ax.set_ylabel('optimal inference depth $K^*$')
ax.set_title(f'Pretrained DDPM ({n_params/1e6:.1f}M params) under the K-sweep protocol')
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(DRIVE_DIR / f'ddpm_kstar.{ext}', dpi=180)
print('figure saved to Drive:', DRIVE_DIR / 'ddpm_kstar.pdf')
plt.show()

## Bring the results home

1. Copy `AAAI_kstar/ddpm_baseline/ddpm_results.json` and `ddpm_kstar.pdf/png` from Drive
   into the repo at `outputs/theory/`.
2. Interpretation:
   - **V1 \u03b1 \u2248 1.3\u20131.45 with R\u00b2 > 0.95** \u2192 the law spans 5.8K\u219235.7M params and
     trained-from-scratch\u2192production-pretrained models. Goes straight into the
     universality figure/table (Section 4.2) and retires reviewer M1.
   - **V1 holds but V2 deviates (or vice versa)** \u2192 conditioning protocol matters; report
     both, discuss in the mechanism section (the fixed-conditioning model is the literal
     analog of a single-\u03c3 energy; noise-matched conditioning changes the effective
     operator per \u03c3).
   - **Neither fits a power law** \u2192 honest boundary result: the law is a property of the
     *fixed-operator* iterative class, not of noise-conditional models \u2014 still a paper-grade
     finding for the phase-boundary section, NOT a failure of the core claim.
3. Log the run in `REGISTERED_CLAIMS.md` with the JSON path (every claim ships with its log).